# **Feature Engineering Project**

## **Introduction**

For this project, we will use the "House Prices - Advanced Regression Techniques" dataset from Kaggle to perform feature engineering and prepare the dataset for building a machine learning model.

## **1. Load the dataset**

In [93]:
# Importr libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [68]:
# Load the dataset

house_data = pd.read_csv("train.csv")

# Display first 10 rows
house_data.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [69]:
house_data.shape

(1460, 81)

## **2. Handling Missing Values**

### **2.1 Inspect the dataset for missing values**

In [70]:
# Identify missing values

missing_values = house_data.isnull().sum()

# Show only columns containing missing values

missing_values[missing_values > 0].sort_values(ascending=False)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

### **2.2 Split data into training and test sets**

To prevent data leakage, all feature engineering methods, including missing-value imputation, feature scaling, and categorical encoding, will be fitted only on the training data, then use the fitted methods to transform the test data.

In [71]:
# Identify features and target

X = house_data.drop(columns=["SalePrice"])
y = house_data["SalePrice"]

# Split 80% training and 20% testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42)

### **2.2 Separate numerical columns and categorical columns**

In [72]:
# Identify numerical columns AFTER splitting

numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]).columns

# Identify categorical columns

categorical_cols = X_train.select_dtypes(
    include=["string"]).columns

### **2.3 Impute numerical missing values with the median**

In [73]:
num_imputer = SimpleImputer(strategy="median")

X_train[numerical_cols] = num_imputer.fit_transform(
    X_train[numerical_cols])

X_test[numerical_cols] = num_imputer.transform(
    X_test[numerical_cols])

### **2.4 Impute categorical missing values**

In [74]:
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train[categorical_cols] = cat_imputer.fit_transform(
    X_train[categorical_cols])

X_test[categorical_cols] = cat_imputer.transform(
    X_test[categorical_cols])

## **3. Feature Engineering**

### **3.1 Create a new feature called ‘TotalSF‘**

TotalSF represents the total square footage by combining basement, first-floor, and second-floor areas.

In [75]:
# Create TotalSF feature

X_train["TotalSF"] = (
    X_train["TotalBsmtSF"]
    + X_train["1stFlrSF"]
    + X_train["2ndFlrSF"])

X_test["TotalSF"] = (
    X_test["TotalBsmtSF"]
    + X_test["1stFlrSF"]
    + X_test["2ndFlrSF"])

In [76]:
X_train["TotalSF"].head()

254     2628.0
1066    2370.0
638     1592.0
799     2499.0
380     2717.0
Name: TotalSF, dtype: float64

### **3.2 Create ‘Age‘ feature**

In [77]:
# Create Age feature

X_train["Age"] = X_train["YrSold"] - X_train["YearBuilt"]
X_test["Age"] = X_test["YrSold"] - X_test["YearBuilt"]

In [78]:
X_train["Age"]

254     53.0
1066    16.0
638     98.0
799     70.0
380     86.0
        ... 
1095     1.0
1130    81.0
1294    51.0
860     89.0
1126     2.0
Name: Age, Length: 1168, dtype: float64

## **4. Feature Scaling**

### **4.1 Identify numerical values**

In [79]:
# Identify the numerical features in the dataset

numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]).columns

### **4.2 Apply Min-Max scaling**

In [80]:
scaler = MinMaxScaler()

# Fit and transform training data

X_train[numerical_cols] = scaler.fit_transform(
    X_train[numerical_cols])

# Transform test data

X_test[numerical_cols] = scaler.transform(
    X_test[numerical_cols])

In [81]:
X_train[numerical_cols].head()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,TotalSF,Age
254,0.174092,0.000000,0.167808,0.033186,0.444444,0.625,0.615942,0.116667,0.000000,0.163359,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.454545,1.00,0.200911,0.389706
1066,0.730637,0.235294,0.130137,0.030555,0.555556,0.750,0.876812,0.733333,0.000000,0.000000,...,0.073126,0.000000,0.0,0.0,0.0,0.0,0.363636,0.75,0.178315,0.117647
638,0.437286,0.058824,0.157534,0.034948,0.444444,0.750,0.275362,0.000000,0.000000,0.000000,...,0.000000,0.297101,0.0,0.0,0.0,0.0,0.363636,0.50,0.110177,0.720588
799,0.547635,0.176471,0.133562,0.027577,0.444444,0.750,0.471014,0.000000,0.182874,0.100815,...,0.000000,0.478261,0.0,0.0,0.0,0.0,0.454545,0.25,0.189613,0.514706
380,0.260452,0.176471,0.099315,0.017294,0.444444,0.625,0.376812,0.000000,0.000000,0.038625,...,0.000000,0.438406,0.0,0.0,0.0,0.0,0.363636,1.00,0.208706,0.632353


## **5. Encoding Categorical Features**

### **5.1 Identify categorical features**

In [82]:
categorical_cols = X_train.select_dtypes(
    include=["string"]).columns

### **5.2 Apply One-Hot Encoding**

In [88]:
# Apply One-Hot encoding

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False)

# Fit on training data
X_train_encoded = encoder.fit_transform(
    X_train[categorical_cols])

# Only transform test data
X_test_encoded = encoder.transform(
    X_test[categorical_cols])

In [89]:
# Get encoded column names
encoded_cols = encoder.get_feature_names_out(categorical_cols)

# Convert encoded arrays to DataFrames
X_train_cat = pd.DataFrame(
    X_train_encoded,
    columns=encoded_cols,
    index=X_train.index)

X_test_cat = pd.DataFrame(
    X_test_encoded,
    columns=encoded_cols,
    index=X_test.index)

In [ ]:
# Keep numerical/preprocessed features

X_train_num = X_train.drop(columns=categorical_cols)
X_test_num = X_test.drop(columns=categorical_cols)

# Combine numerical + encoded categorical features

X_train_final = pd.concat(
    [X_train_num, X_train_cat], axis=1)

X_test_final = pd.concat(
    [X_test_num, X_test_cat], axis=1)

## **6. Model Training**

In [91]:
# Train the Linear Regression model

linear_model = LinearRegression()

linear_model.fit(X_train_final, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](288,)","[ 786.72, 2043.45, 2634.87,...,-9421.29,-4823.66, 6122.19]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](288,)","['Id','MSSubClass','LotFrontage',...,'SaleCondition_Family', 'SaleCondition_Normal','SaleCondition_Partial']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-2.269e+05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,288
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(240)


In [ ]:
# Predictions on training data

y_train_pred = linear_model.predict(X_train_final)

# Predictions on test data

y_test_pred = linear_model.predict(X_test_final)

In [94]:
# Calculate MSE

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

print("Training MSE:", train_mse)
print("Test MSE:", test_mse)

Training MSE: 381383614.63929
Test MSE: 868893813.4672074


### **Insights**:

The Linear Regression model achieved a training MSE of 381,383,614.64 and a test MSE of 868,893,813.47. The test MSE is  higher than the training MSE, indicating that the model performs better on the training data than on unseen data. This suggests some degree of overfitting or limited generalization. Further model improvement could involve feature selection, regularization, or testing more advanced regression models.